# Framing Reannotation: Improved Prompt + Conflict Resolution

This notebook:
1. Loads `annotation_100k.csv` (original labels = `label`)
2. Re-annotates every row with an improved 3-class prompt (label2)
3. Identifies conflicts where `label` and `label2` disagree
4. Sends conflicts to the LLM a third time with a conflict-resolution prompt to produce `label3`
5. Merges everything into a final dataset

## Setup

In [1]:
import pandas as pd
import requests
import os
import time
from dotenv import load_dotenv

In [2]:

load_dotenv()
api_key = os.getenv("CAC_API_KEY")

HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "ministral-3-14b"

print(f"API key loaded: {'YES' if api_key else 'NO'}")
print(f"Host: {HOST} | Model: {MODEL}")

API key loaded: YES
Host: https://maki.uni-mannheim.de/v1 | Model: ministral-3-14b


## Annotation Instructions

Three labels: SECURITY_NEGATIVE, SECURITY_POSITIVE, NOT_SECURITY_CRIME.

Key improvements over the original prompt:
- Victim rule is explicit and prominent
- War/military reporting between states is excluded
- Historical violence contexts excluded
- Debunked frames excluded
- No group labels passed to the model

In [3]:
instructions = """Sie sind ein neutraler Annotationsassistent fuer ein wissenschaftliches Projekt zur Inhaltsanalyse deutscher Zeitungsartikel.

Ihre Aufgabe ist es, einen einzelnen Zeitungsabsatz zu klassifizieren. Beurteilen Sie ausschliesslich den vorliegenden Text.

Es gibt genau drei Labels:

SECURITY_NEGATIVE: Der Absatz rahmt eine Gruppe, Massnahme oder ein Ereignis explizit als Gefahr, Bedrohung, Kriminalitaetsquelle, Instabilitaetsfaktor, Gewaltrisiko oder oeffentliches Sicherheitsproblem. Die Gruppe wird als Verursacher oder Quelle des Problems dargestellt.

SECURITY_POSITIVE: Eine Sicherheitsmassnahme, Politik oder Institution wird explizit als erfolgreiche Bewaeltigung einer Bedrohung oder als Schutz der Oeffentlichkeit dargestellt. Beispiele: Polizei verhaftet Taeter erfolgreich, Abschiebung von Kriminellen wird als Erfolg gerahmt, Grenzschutz stellt Ordnung wieder her. Nur vergeben wenn eine positive Sicherheitsleistung explizit gerahmt wird. NICHT bei internationaler Kriegsberichterstattung zwischen Staaten.

NOT_SECURITY_CRIME: Alles andere. Dieses Label vergeben Sie insbesondere wenn:
- Die genannte Gruppe das OPFER von Gewalt, Kriminalitaet, Verfolgung oder staatlicher Repression ist
- Der Absatz ueber internationalen Militaerkonflikt zwischen Staaten berichtet (z.B. Russland greift Ukraine an, NATO-Operationen, Waffenlieferungen zwischen Laendern) ohne Bezug zu Minderheiten in Deutschland
- Kriminalitaet auf strukturelle Ursachen wie Armut oder psychische Erkrankung zurueckgefuehrt wird, nicht auf die Gruppe selbst
- Der Absatz einen Sicherheitsrahmen explizit in Frage stellt oder widerlegt
- Es sich um historische Gewalt handelt, bei der die genannte Gruppe Opfer war (z.B. Holocaust, Pogrome, Kolonialgewalt)
- Polizei, Gerichte oder Grenzen neutral erwaehnt werden ohne klare Bedrohungszuschreibung an die Gruppe
- Die Gruppe Opfer von rechtsextremer Gewalt, Rassismus oder Polizeigewalt ist

Entscheidungsregeln:
1. Fragen Sie sich zuerst: Ist die genannte Gruppe der Taeter oder das Opfer? Opfer = NOT_SECURITY_CRIME
2. Handelt es sich um Kriegsberichterstattung zwischen Staaten ohne Bezug zu Minderheiten in Deutschland? = NOT_SECURITY_CRIME
3. Wird ein Sicherheitsrahmen explizit bestritten oder als Desinformation entlarvt? = NOT_SECURITY_CRIME
4. Wird eine Sicherheitsmassnahme als Erfolg gerahmt (Verhaftung, Abschiebung, Grenzschutz)? = SECURITY_POSITIVE
5. Nur wenn die Gruppe klar als Quelle von Bedrohung oder Kriminalitaet dargestellt wird: SECURITY_NEGATIVE"""

reminder = "Klassifizieren Sie den Absatz als SECURITY_NEGATIVE, SECURITY_POSITIVE oder NOT_SECURITY_CRIME. Vergeben Sie SECURITY_NEGATIVE nur wenn die Gruppe explizit als Quelle von Bedrohung oder Kriminalitaet dargestellt wird. Vergeben Sie SECURITY_POSITIVE nur wenn eine Sicherheitsmassnahme als Erfolg gerahmt wird. Wenn die Gruppe Opfer ist oder es sich um internationale Kriegsberichterstattung handelt: NOT_SECURITY_CRIME."

conflict_instructions = """Sie sind ein neutraler Annotationsexperte. Zwei unabhaengige Annotationslaeufe haben fuer den folgenden Absatz unterschiedliche Labels vergeben.

Label aus Lauf 1: {label1}
Label aus Lauf 2: {label2}

Ihre Aufgabe ist es, das korrekte Label zu bestimmen. Es gibt drei moegliche Labels:

SECURITY_NEGATIVE: Gruppe wird explizit als Quelle von Bedrohung oder Kriminalitaet dargestellt.
SECURITY_POSITIVE: Eine Sicherheitsmassnahme wird explizit als Erfolg gerahmt (Verhaftung, Abschiebung, Grenzschutz).
NOT_SECURITY_CRIME: Alles andere, insbesondere wenn die Gruppe Opfer ist, es sich um Kriegsberichterstattung zwischen Staaten handelt, oder kein klarer Sicherheitsrahmen vorliegt.

Schluesselfragen:
- Ist die genannte Gruppe Taeter oder Opfer?
- Handelt es sich um internationale Kriegsberichterstattung ohne Bezug zu Minderheiten in Deutschland?
- Wird die Gruppe klar als Bedrohungsquelle bezeichnet?
- Wird eine Sicherheitsmassnahme als Erfolg gerahmt?

Begruenden Sie Ihre Entscheidung in einem Satz und vergeben Sie dann das finale Label."""

print("Instructions loaded.")

Instructions loaded.


## Core Functions

In [4]:
VALID_LABELS = ["SECURITY_NEGATIVE", "SECURITY_POSITIVE", "NOT_SECURITY_CRIME"]


def call_api(messages, temperature=0.0):
    payload = {
        "model": MODEL,
        "temperature": temperature,
        "messages": messages
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    for attempt in range(3):
        try:
            r = requests.post(
                f"{HOST}/chat/completions",
                json=payload,
                headers=headers,
                timeout=120
            )
            if r.status_code == 200:
                return r.json()["choices"][0]["message"]["content"].strip()
            print(f"  [attempt {attempt+1}] status {r.status_code}")
        except Exception as e:
            print(f"  [attempt {attempt+1}] error: {e}")
            time.sleep(5)
    return "ERROR"


def parse_label(raw):
    u = raw.upper()
    # check NOT first since SECURITY is a substring of the other two
    if "NOT_SECURITY_CRIME" in u or "NOT SECURITY CRIME" in u:
        return "NOT_SECURITY_CRIME"
    elif "SECURITY_NEGATIVE" in u or "SECURITY NEGATIVE" in u:
        return "SECURITY_NEGATIVE"
    elif "SECURITY_POSITIVE" in u or "SECURITY POSITIVE" in u:
        return "SECURITY_POSITIVE"
    return "UNCLEAR"


def parse_explanation(raw):
    for line in raw.split("\n"):
        if line.lower().startswith("explanation:"):
            return line[len("explanation:"):].strip()
    return raw[:300]


def annotate(text, temperature=0.0):
    prompt = (
        f"{instructions}\n\n"
        f"Annotieren Sie jetzt diesen Absatz:\n\n{text}\n\n"
        f"{reminder}\n\n"
        "Antworten Sie genau in diesem Format:\n"
        "Label: <SECURITY_NEGATIVE oder SECURITY_POSITIVE oder NOT_SECURITY_CRIME>\n"
        "Explanation: <ein Satz>"
    )
    messages = [
        {"role": "system", "content": "Sie sind ein Experte fuer Inhaltsanalyse. Antworten Sie immer im angegebenen Format."},
        {"role": "user",   "content": prompt}
    ]
    return call_api(messages, temperature)


def resolve_conflict(text, label1, label2):
    system = conflict_instructions.format(label1=label1, label2=label2)
    prompt = (
        f"Hier ist der Absatz:\n\n{text}\n\n"
        "Antworten Sie in diesem Format:\n"
        "Explanation: <ein Satz>\n"
        "Label: <SECURITY_NEGATIVE oder SECURITY_POSITIVE oder NOT_SECURITY_CRIME>"
    )
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": prompt}
    ]
    return call_api(messages, temperature=0.0)


print("Functions loaded.")

Functions loaded.


## Load Data

In [5]:
df = pd.read_csv(
    "results/annotation_100k.csv",
    usecols=["article_id", "pub", "par_index", "group", "text_block", "label"]
)

# clean up any UNCLEAR rows from original run
df["label"] = df["label"].replace({"UNCLEAR": "NOT_SECURITY_CRIME"})

df = df.reset_index(drop=True)
print(f"Loaded {len(df)} rows")
print(df["label"].value_counts())

Loaded 94572 rows
label
NOT_SECURITY_CRIME    84931
SECURITY_NEGATIVE      8084
SECURITY_POSITIVE      1557
Name: count, dtype: int64


In [8]:
df = df[df["label"].isin(["SECURITY_NEGATIVE", "SECURITY_POSITIVE"])]
print(df["label"].value_counts())

label
SECURITY_NEGATIVE    8084
SECURITY_POSITIVE    1557
Name: count, dtype: int64


## Step 1: Re-annotate All Rows (label2)

Saves every 500 rows. If interrupted, rerun and it resumes from where it left off.

In [10]:
LABEL2_PATH      = "results/annotation_label2.csv"
CHECKPOINT_EVERY = 500

os.makedirs("results", exist_ok=True)

if os.path.exists(LABEL2_PATH):
    done = pd.read_csv(LABEL2_PATH)
    done_ids = set(zip(done["article_id"].astype(str), done["par_index"].astype(str)))
    print(f"Resuming: {len(done)} rows done, {len(df) - len(done)} remaining.")
else:
    done = pd.DataFrame()
    done_ids = set()
    print(f"Starting fresh: {len(df)} rows to annotate.")

buffer     = []
total_done = len(done_ids)
n_total    = len(df)
start_time = time.time()

for i, row in df.iterrows():
    key = (str(row["article_id"]), str(row["par_index"]))
    if key in done_ids:
        continue

    raw         = annotate(str(row["text_block"]))
    label2      = parse_label(raw)
    explanation = parse_explanation(raw)

    buffer.append({
        "article_id":   row["article_id"],
        "pub":          row["pub"],
        "par_index":    row["par_index"],
        "group":        row["group"],
        "text_block":   row["text_block"],
        "label":        row["label"],
        "label2":       label2,
        "explanation2": explanation,
    })
    done_ids.add(key)
    total_done += 1

    if total_done % CHECKPOINT_EVERY == 0:
        chunk = pd.DataFrame(buffer)
        write_header = not os.path.exists(LABEL2_PATH) or total_done == CHECKPOINT_EVERY
        chunk.to_csv(LABEL2_PATH, mode="a", header=write_header, index=False)
        done   = pd.concat([done, chunk], ignore_index=True)
        buffer = []

        elapsed   = time.time() - start_time
        remaining = (elapsed / total_done) * (n_total - total_done)
        conflicts = (done["label"] != done["label2"]).sum()
        print(
            f"  [{total_done:>6}/{n_total}]  "
            f"conflicts so far: {conflicts}  "
            f"ETA: {remaining/3600:.1f}h"
        )

if buffer:
    chunk = pd.DataFrame(buffer)
    write_header = not os.path.exists(LABEL2_PATH)
    chunk.to_csv(LABEL2_PATH, mode="a", header=write_header, index=False)
    done = pd.concat([done, chunk], ignore_index=True)

print(f"\nFinished label2. Total: {len(done)}")
print(done["label2"].value_counts())

Resuming: 2500 rows done, 7141 remaining.
  [  3000/9641]  conflicts so far: 569  ETA: 0.2h
  [  3500/9641]  conflicts so far: 1060  ETA: 0.2h
  [  4000/9641]  conflicts so far: 1534  ETA: 0.3h


KeyboardInterrupt: 

## Step 2: Find Conflicts

In [11]:
df2 = pd.read_csv(LABEL2_PATH)

# only rows where label2 is a valid label
valid = df2[df2["label2"].isin(VALID_LABELS)].copy()

conflicts  = valid[valid["label"] != valid["label2"]].copy()
agreements = valid[valid["label"] == valid["label2"]].copy()

print(f"Total valid rows:  {len(valid)}")
print(f"Agreements:        {len(agreements)} ({len(agreements)/len(valid)*100:.1f}%)")
print(f"Conflicts:         {len(conflicts)} ({len(conflicts)/len(valid)*100:.1f}%)")
print("\nConflict breakdown (label1 -> label2):")
print(conflicts.groupby(["label", "label2"]).size().reset_index(name="count"))

Total valid rows:  4000
Agreements:        2466 (61.7%)
Conflicts:         1534 (38.4%)

Conflict breakdown (label1 -> label2):
                label              label2  count
0  NOT_SECURITY_CRIME   SECURITY_POSITIVE      1
1   SECURITY_NEGATIVE  NOT_SECURITY_CRIME   1228
2   SECURITY_NEGATIVE   SECURITY_POSITIVE     21
3   SECURITY_POSITIVE  NOT_SECURITY_CRIME    284


In [12]:
# spot check some conflicts before resolving
conflicts[["group", "pub", "label", "label2", "explanation2", "text_block"]].head(20)

,group,pub,label,label2,explanation2,text_block
3,BGR,MAZ,SECURITY_NEGATIVE,NOT_SECURITY_CRIME,Der Absatz beschreibt strukturelle Probleme (i...,Und genau darauf dürfte es hinauslaufen. Denn ...
15,BGR,MAZ,SECURITY_POSITIVE,NOT_SECURITY_CRIME,**Label:** NOT_SECURITY_CRIME\n\n**Explanation...,"Und so scheint denn erst einmal zu verpuffen, ..."
39,BGR,MOPO,NOT_SECURITY_CRIME,SECURITY_POSITIVE,Die Bundespolizei wird als erfolgreiche Sicher...,\n\nAm Montagmorgen wurde am Flughafen Hamburg...
40,BGR,KN,SECURITY_NEGATIVE,NOT_SECURITY_CRIME,Die Gruppe [Gruppe 1] wird als **Opfer** des G...,Der Geiselnehmer des Überfalls auf einen Apple...
41,BGR,KN,SECURITY_NEGATIVE,NOT_SECURITY_CRIME,**Label:** NOT_SECURITY_CRIME\n**Explanation:*...,"Die Geisel, ein [Gruppe 1], blieb unverletzt, ..."
45,BGR,KN,SECURITY_NEGATIVE,SECURITY_POSITIVE,Der Absatz rahmt die **Festnahmebemühungen des...,"Die US-Bundespolizei FBI hat die als ""Krypto-Q..."
52,BGR,KN,SECURITY_NEGATIVE,NOT_SECURITY_CRIME,Der Absatz beschreibt strukturelle Benachteili...,Weil die Betroffenen gar nicht oder kaum über ...
57,BGR,LVZ,SECURITY_NEGATIVE,NOT_SECURITY_CRIME,Der Absatz beschreibt die Gruppe **[Gruppe 1]*...,Der Geiselnehmer des Überfalls auf einen Apple...
93,FRA,MAZ,SECURITY_NEGATIVE,NOT_SECURITY_CRIME,Der Absatz berichtet über einen **internationa...,Und es droht neue Gewalt. Wie konkret Gewalt w...
108,FRA,MAZ,SECURITY_NEGATIVE,NOT_SECURITY_CRIME,Der Absatz berichtet über internationale Milit...,PARIS / BERLIN (dpa). Frankreich beendet seine...


## Step 3: Resolve Conflicts (label3)

For every conflicting row, the LLM receives both previous labels and the text and is asked to adjudicate.

In [13]:
LABEL3_PATH = "results/annotation_label3_conflicts.csv"

if os.path.exists(LABEL3_PATH):
    done3 = pd.read_csv(LABEL3_PATH)
    done3_ids = set(zip(done3["article_id"].astype(str), done3["par_index"].astype(str)))
    print(f"Resuming: {len(done3)} resolved, {len(conflicts) - len(done3)} remaining.")
else:
    done3 = pd.DataFrame()
    done3_ids = set()
    print(f"Starting conflict resolution: {len(conflicts)} conflicts.")

buffer3     = []
n_conflicts = len(conflicts)

for i, (_, row) in enumerate(conflicts.iterrows()):
    key = (str(row["article_id"]), str(row["par_index"]))
    if key in done3_ids:
        continue

    raw    = resolve_conflict(str(row["text_block"]), row["label"], row["label2"])
    label3 = parse_label(raw)
    expl3  = parse_explanation(raw)

    buffer3.append({
        "article_id":   row["article_id"],
        "pub":          row["pub"],
        "par_index":    row["par_index"],
        "group":        row["group"],
        "text_block":   row["text_block"],
        "label":        row["label"],
        "label2":       row["label2"],
        "label3":       label3,
        "explanation3": expl3,
    })
    done3_ids.add(key)

    if (i + 1) % 100 == 0:
        chunk3 = pd.DataFrame(buffer3)
        write_header = not os.path.exists(LABEL3_PATH) or i < 100
        chunk3.to_csv(LABEL3_PATH, mode="a", header=write_header, index=False)
        done3   = pd.concat([done3, chunk3], ignore_index=True)
        buffer3 = []
        print(f"  [{i+1}/{n_conflicts}] resolved")

if buffer3:
    chunk3 = pd.DataFrame(buffer3)
    write_header = not os.path.exists(LABEL3_PATH)
    chunk3.to_csv(LABEL3_PATH, mode="a", header=write_header, index=False)
    done3 = pd.concat([done3, chunk3], ignore_index=True)

print(f"\nDone. Total resolved: {len(done3)}")
print(done3["label3"].value_counts())

Starting conflict resolution: 1534 conflicts.
  [100/1534] resolved
  [200/1534] resolved
  [300/1534] resolved
  [400/1534] resolved
  [500/1534] resolved
  [600/1534] resolved
  [700/1534] resolved
  [800/1534] resolved
  [900/1534] resolved
  [1000/1534] resolved
  [1100/1534] resolved
  [1200/1534] resolved
  [1300/1534] resolved
  [1400/1534] resolved
  [1500/1534] resolved

Done. Total resolved: 1534
label3
NOT_SECURITY_CRIME    1124
SECURITY_NEGATIVE      308
SECURITY_POSITIVE      102
Name: count, dtype: int64


## Step 4: Merge into Final Dataset

Logic:
- label == label2: use that as final_label
- label != label2: use label3
- label2 is UNCLEAR: fall back to label

In [ ]:
FINAL_PATH = "results/annotation_final.csv"

df2 = pd.read_csv(LABEL2_PATH)
df3 = pd.read_csv(LABEL3_PATH)

df3_lookup = df3.set_index(["article_id", "par_index"])["label3"].to_dict()
df3_expl   = df3.set_index(["article_id", "par_index"])["explanation3"].to_dict()


def assign_final(row):
    key = (row["article_id"], row["par_index"])
    if row["label2"] == "UNCLEAR":
        return row["label"], "label2 unclear, fell back to label1"
    elif row["label"] == row["label2"]:
        return row["label"], "agreement"
    else:
        l3   = df3_lookup.get(key, row["label2"])
        expl = df3_expl.get(key, "")
        return l3, expl


results = df2.apply(assign_final, axis=1, result_type="expand")
df2["final_label"]       = results[0]
df2["final_explanation"] = results[1]

df2.to_csv(FINAL_PATH, index=False)

print(f"Final dataset saved: {len(df2)} rows")
print("\nFinal label distribution:")
total = len(df2)
for label, count in df2["final_label"].value_counts().items():
    print(f"  {label}: {count} ({count/total*100:.1f}%)")

## Step 5: Inspect Results

In [ ]:
final = pd.read_csv(FINAL_PATH)

valid = final[final["label2"].isin(VALID_LABELS)]
agreement_rate = (valid["label"] == valid["label2"]).mean()
print(f"Agreement between label1 and label2: {agreement_rate:.1%}")

df3 = pd.read_csv(LABEL3_PATH)
print(f"\nConflicts resolved: {len(df3)}")
print(f"label3 sided with label1: {(df3['label3'] == df3['label']).sum()}")
print(f"label3 sided with label2: {(df3['label3'] == df3['label2']).sum()}")
print(f"label3 sided with neither: {((df3['label3'] != df3['label']) & (df3['label3'] != df3['label2'])).sum()}")
print(f"label3 UNCLEAR:            {(df3['label3'] == 'UNCLEAR').sum()}")

In [ ]:
# distribution by outlet
final.groupby("pub")["final_label"].value_counts(normalize=True).unstack().fillna(0).round(3)

In [ ]:
# distribution by group, sorted by SECURITY_NEGATIVE rate
(
    final
    .groupby("group")["final_label"]
    .value_counts(normalize=True)
    .unstack()
    .fillna(0)
    .round(3)
    .sort_values("SECURITY_NEGATIVE", ascending=False)
    .head(25)
)

In [ ]:
# spot check resolved conflicts
df3[["group", "pub", "label", "label2", "label3", "explanation3", "text_block"]].head(20)